In [1]:
import re
import numpy as np
import pandas as pd
LOG = open("lab04_server_log.txt", encoding="utf-8").read().splitlines()
products = pd.read_csv("lab04_products.csv", dtype=str, keep_default_na=False)
print(len(LOG), "log lines |", products.shape, "product records")

117 log lines | (60, 5) product records


In [2]:
products

,sku,name,dims,contact,description
0,AB-1042-XY,Blue Widget,12x8x3 in,orders@northwind.example.com,Ships flat. Weighs 2.4 lb.
1,CD-2087-QR,Steel Bracket,12 x 8 x 3 in,(202) 555-0143,Weighs 3 lb; fits a 12in shelf.
2,EF-3311-ZW,Cotton Tote,30.5cm x 20cm x 7.6cm,202-555-0198,Metric packaging. Weighs 1.1 kg.
3,GH-4520-LM,Ceramic Mug,9x9x9in,support@acme.example.org,Cube. Weighs 0.75 lb.
4,IJ-5077-TT,Oak Shelf,18 x 12 x 4 in,+1 202 555 0166,"Weighs 5.2 lb, ships in 2 boxes."
5,ab-6001-yz,Brass Hinge,6x4x2 in,sales@acme.example.org,Lowercase SKU -- entry error.
6,KL-7,Linen Runner,10x10x2 in,202.555.0177,Short SKU -- entry error.
7,MN-8123-PQ,Copper Wire,,hello@example,No dims recorded. Bad email.
8,OP-9044-RS,Glass Vase,24x18x6 in,,No contact on file. Weighs 11 lb.
9,QR-1150-UV,Pine Crate,1.5 x 1.5 x 0.25 in,1-202-555-0102,Tiny. Weighs 0.05 lb.


In [3]:
LOG

['10.0.0.7 - - [11/Mar/2026:08:04:16 -0500] "GET /catalog HTTP/1.1" 200 46290 "https://www.google.com/" "Googlebot/2.1 (+http://www.google.com/bot.html)"',
 '192.0.2.88 - - [11/Mar/2026:08:09:10 -0500] "GET /robots.txt HTTP/1.1" 404 33681 "https://example.edu/catalog" "Googlebot/2.1 (+http://www.google.com/bot.html)"',
 '203.0.113.19 - - [11/Mar/2026:08:11:00 -0500] "POST /api/v2/price HTTP/1.1" 200 27783 "-" "curl/8.4.0"',
 '192.0.2.88 - - [11/Mar/2026:08:12:55 -0500] "GET /admin HTTP/1.1" 403 23698 "https://www.google.com/" "python-requests/2.32.3"',
 '172.16.4.201 - - [11/Mar/2026:08:18:20 -0500] "POST /catalog HTTP/1.1" 200 11256 "https://example.edu/" "python-requests/2.32.3"',
 '192.168.1.24 - - [11/Mar/2026:08:22:14 -0500] "GET /cart HTTP/1.1" 200 25383 "https://example.edu/" "curl/8.4.0"',
 '198.51.100.42 - - [11/Mar/2026:08:27:17 -0500] "POST /static/app.css HTTP/1.1" 200 9707 "-" "curl/8.4.0"',
 '203.0.113.19 - - [11/Mar/2026:08:33:27 -0500] "GET /search?q=blue+widget HTTP/1.

# 1

In [4]:
# Q1.1
print(re.match(r"\d{3}", "HTTP/1.1 404 Not Found"))
print(re.fullmatch(r"\d{3}", "HTTP/1.1 404 Not Found"))
print(re.search(r"\d{3}", "HTTP/1.1 404 Not Found"))

# Match and full match print None, while search actually gives the match "404"

print(re.match(r"\d{3}", "404abc"))
# This matters because match will only take the first part of the string

None
None
<re.Match object; span=(9, 12), match='404'>
<re.Match object; span=(0, 3), match='404'>


In [5]:
# Q1.2
print("findall")
for i in range(10):
    print(re.findall(r"\d{3}", LOG[i]))
print("finditer")
for i in range(10):
    print(re.finditer(r"\d{3}", LOG[i]))
    
# findall gives a list of every 3 digit number, finditer gives an iterator object of every 3 digit number.
# You could need finditer if you want to iterate over every number

# I predict this will print each digit in the string
print(re.findall(r"(\d)(\d)(\d)", "a 404 b 500"))
# It printed each digit in the string, but organized together

findall
['202', '050', '200', '462']
['192', '202', '050', '404', '336']
['203', '113', '202', '050', '200', '277']
['192', '202', '050', '403', '236']
['172', '201', '202', '050', '200', '112']
['192', '168', '202', '050', '200', '253']
['198', '100', '202', '050', '200', '970']
['203', '113', '202', '050', '200', '277']
['203', '113', '202', '050', '200', '315', '201', '001', '128']
['200', '370', '733', '202', '050', '200', '472']
finditer
[('4', '0', '4'), ('5', '0', '0')]


In [6]:
# Q1.3
pattern = r"(?P<method>[A-Z]+)\s+(?P<path>/\S*)"
m = re.search(pattern, "GET /catalog/item?id=1042 HTTP/1.1")

print(m.group(0))
print(m.group(1))
print(m.group("path"))
print(m.groupdict())

# Thery are worth the extra characters for better readability

GET /catalog/item?id=1042
GET
/catalog/item?id=1042
{'method': 'GET', 'path': '/catalog/item?id=1042'}


In [7]:
# Q1.4
print(repr(r"\b"))
print(repr("\b"))
re.findall("\bcat\b", "cat concatcats")

# Without the r, we either get the actual character or we get nothing (empty list)

'\\b'
'\x08'


[]

# 2

In [8]:
# Q2.1
SKU = re.compile(r"^[A-Z]{2}-\d{4}-[A-Z]{2}$")

count = 0
valid = 0
for x in products["sku"]:
    count += 1
    if SKU.fullmatch(x):
        valid += 1
        
print(f"valid: {valid}")
print(f"invalid: {count-valid}")

valid: 48
invalid: 12


In [9]:
# Q2.2
EMAIL = re.compile(r"""
    ^
    [^@\s]+
    @
    [^@\s]+
    \.
    [A-Za-z]{2,}
    $
""", re.VERBOSE)

PHONE = re.compile(r"""
    ^
    (?:
        \d{3}-\d{3}-\d{4}
      | \d{3}\.\d{3}\.\d{4}
      | \(\d{3}\)\s*\d{3}-\d{4}
      | \+1\s\d{3}\s\d{3}\s\d{4}
      | 1-\d{3}-\d{3}-\d{4}
    )
    $
""", re.VERBOSE)

email_count = 0
phone_count = 0
none_count = 0
neither_count = 0

for x in products["contact"]:
    if EMAIL.fullmatch(x):
        email_count += 1
    elif PHONE.fullmatch(x):
        phone_count += 1
    elif x == "":
        none_count += 1
    else:
        neither_count += 1

print(email_count)
print(phone_count)
print(none_count)
print(neither_count)

# I chose to accept the following for phone numbers:
# 1. +1 country code
# 2. parenthesised area
# 3. dot separated
# 4. hyphen separated
# 5. hyphen separated +1

18
30
6
6


In [10]:
# Q2.3
SKU = re.compile(r"^[A-Z]{2}-\d{4}-[A-Z]{2}$", re.IGNORECASE)

count = 0
valid = 0
for x in products["sku"]:
    count += 1
    if SKU.fullmatch(x):
        valid += 1
        
print(f"valid: {valid}")
print(f"invalid: {count-valid}")

# The flag allows for some rows to be valid, likely if the are lowercase in some places. If a collegue said that
# I would ask if it is ok for there to be both uppercase and lowercase in one. If not, then that would not be the
# best approach, and implement a way ot check that all characters are uniform case

valid: 54
invalid: 6


# 3

In [11]:
# Q3.1
LOG_regex = re.compile(r"""
    ^
    (?P<ip>\S+)                     # client IP
    \s+-\s+-\s+                     # two unused identity fields, always -
    \[(?P<ts>[^\]]+)\]              # timestamp inside [], any char but the closing ]
    \s+"                            # opening quote of the request line
    (?P<method>[A-Z]+)              # method: capitals only
    \s+
    (?P<path>\S+)                   # path: no spaces, stops before the protocol
    \s+HTTP/\d\.\d                  # protocol, matched but not captured
    "                               # closing quote of the request line
    \s+(?P<status>\d{3})            # status code: exactly three digits
    \s+(?P<size>\d+|-)              # response size, or - when there is no body
    \s+"(?P<referer>[^"]*)"         # referer, may be empty or "-"
    \s+"(?P<agent>[^"]*)"           # user-agent, may contain spaces and slashes
    $
""", re.VERBOSE)


count = 0
valid = 0
for line in LOG:
    count += 1
    if LOG_regex.fullmatch(line):
        valid += 1
    else:
        print(line)

print(f"valid: {valid}")
print(f"invalid: {count-valid}")

10.0.0.7 - - [11/Mar/2026:09:03:11 -0500] "GET /catalog HTTP/1.1" 200
203.0.113.19 - - [11/Mar/2026:10:22:47 -0500] "GET /cart HTTP/1.1" 200 1120 "-"
<<< log rotated at 11/Mar/2026:11:00:00 >>>
198.51.100.42 - - 11/Mar/2026:12:14:03 -0500 "GET /health HTTP/1.1" 200 12 "-" "curl/8.4.0"
valid: 113
invalid: 4


In [12]:
# Q3.2
for line in LOG:
    if not LOG_regex.fullmatch(line):
        print(line)

# Only 4: 
# first one is missing 3 fields at the end
# second one is missing the last field
# third one has no request at all
# fourth one is missing square brackets around the timestamp

10.0.0.7 - - [11/Mar/2026:09:03:11 -0500] "GET /catalog HTTP/1.1" 200
203.0.113.19 - - [11/Mar/2026:10:22:47 -0500] "GET /cart HTTP/1.1" 200 1120 "-"
<<< log rotated at 11/Mar/2026:11:00:00 >>>
198.51.100.42 - - 11/Mar/2026:12:14:03 -0500 "GET /health HTTP/1.1" 200 12 "-" "curl/8.4.0"


In [13]:
# Q3.3

rows = []
for line in LOG:
    if LOG_regex.fullmatch(line):
        rows.append(LOG_regex.fullmatch(line).groupdict())

df = pd.DataFrame(rows, columns=["ip","ts","method","path","status","size","referer","agent"])
 
df["status"] = df["status"].astype("int64")
df["bytes"]  = pd.to_numeric(df["size"], errors="coerce")          # "-" -> NaN
df["when"]   = pd.to_datetime(df["ts"], format="%d/%b/%Y:%H:%M:%S %z", utc=True)
 
df = df.drop(columns=["ts", "size"])
df = df[["when", "ip", "method", "path", "status", "bytes", "referer", "agent"]]

print(df)

# Missing byte counts shoudl be NaN, because they are certainly not 0. If we do any operations or anything on them
# we will make incorrect assumptions

                         when             ip method           path  status  \
0   2026-03-11 13:04:16+00:00       10.0.0.7    GET       /catalog     200   
1   2026-03-11 13:09:10+00:00     192.0.2.88    GET    /robots.txt     404   
2   2026-03-11 13:11:00+00:00   203.0.113.19   POST  /api/v2/price     200   
3   2026-03-11 13:12:55+00:00     192.0.2.88    GET         /admin     403   
4   2026-03-11 13:18:20+00:00   172.16.4.201   POST       /catalog     200   
..                        ...            ...    ...            ...     ...   
108 2026-03-11 18:51:20+00:00   192.168.1.24   POST  /api/v2/price     200   
109 2026-03-11 18:52:30+00:00   192.168.1.24    GET        /health     200   
110 2026-03-11 18:56:17+00:00  198.51.100.42    GET          /cart     200   
111 2026-03-11 19:01:57+00:00  198.51.100.42    GET          /cart     200   
112 2026-03-11 19:06:27+00:00   203.0.113.19   HEAD         /admin     403   

       bytes                      referer  \
0    46290.0      

In [14]:
# Q3.4
# a) 
cls = df["status"] // 100
fourxx = int((cls == 4).sum())
fivexx = int((cls == 5).sum())
print(f"a) 4xx: {fourxx} 5xx: {fivexx}")
print()
# b)
print(f"b) {df['path'].value_counts().head(5)}")
print()
# c)
import ipaddress

def is_ipv4(s):
    try:
        ipaddress.IPv4Address(s)
        return True
    except ValueError:
        return False

mask = ~df["ip"].map(is_ipv4)
print(f"c) {int(mask.sum())}")

a) 4xx: 31 5xx: 8

b) /api/v2/price            15
/catalog/item?id=9999    10
/admin                    9
/cart                     9
/search?q=blue+widget     9
Name: path, dtype: int64

c) 4


In [15]:
# Q4.1
DIMS = re.compile(r"""
    ^\s*
    (?P<d1>\d+(?:\.\d+)?)   \s* (?P<u1>cm|mm|in|ft|m)?
    \s* [x×X] \s*
    (?P<d2>\d+(?:\.\d+)?)   \s* (?P<u2>cm|mm|in|ft|m)?
    \s* [x×X] \s*
    (?P<d3>\d+(?:\.\d+)?)   \s* (?P<u3>cm|mm|in|ft|m)?
    \s*$
""", re.VERBOSE)

ex = products["dims"].str.extract(DIMS)

out = pd.DataFrame({
    "d1": pd.to_numeric(ex["d1"]),
    "d2": pd.to_numeric(ex["d2"]),
    "d3": pd.to_numeric(ex["d3"]),
    "unit": ex[["u1", "u2", "u3"]].bfill(axis=1).ffill(axis=1)["u3"],
})
print(out.head(10))
# the 6 that do not parse are empty, and ignore case could be appropriate here if we have units that are
# different cases

     d1    d2    d3 unit
0  12.0   8.0  3.00   in
1  12.0   8.0  3.00   in
2  30.5  20.0  7.60   cm
3   9.0   9.0  9.00   in
4  18.0  12.0  4.00   in
5   6.0   4.0  2.00   in
6  10.0  10.0  2.00   in
7   NaN   NaN   NaN  NaN
8  24.0  18.0  6.00   in
9   1.5   1.5  0.25   in


In [16]:
# Q4.2
UNIT = r"(?:pounds?|lbs?|kilograms?|kgs?|grams?|g|ounces?|oz|mg)"

WEIGHT = re.compile(r"""
    (?<![\d.])
    (?P<value>\d+(?:\.\d+)?)
    \s*
    (?P<unit>pounds?|lbs?|kilograms?|kgs? 
            |grams?|ounces?|oz|mg|kg|g)
    \b
""", re.VERBOSE | re.IGNORECASE)

weights = products["description"].str.extract(WEIGHT)
weights["value"] = pd.to_numeric(weights["value"])
weights["unit"]  = weights["unit"].str.lower()
print(weights.head(10))

   value unit
0   2.40   lb
1   3.00   lb
2   1.10   kg
3   0.75   lb
4   5.20   lb
5    NaN  NaN
6    NaN  NaN
7    NaN  NaN
8  11.00   lb
9   0.05   lb


In [17]:
# Q4.3
REDACT = re.compile(r"""
    (?P<head> # everything up to the last four digits
        \(\d{3}\)\s*\d{3}-
      | \d{3}-\d{3}-
      | \d{3}\.\d{3}\.
      | \+1\s\d{3}\s\d{3}\s
      | 1-\d{3}-\d{3}-
    )
    (?P<last4>\d{4}) # the four digits to hide
    (?!\d)
""", re.VERBOSE)

def redact(m):
    """Replacement callable: gets the match, returns the new text."""
    return m.group("head") + "X" * len(m.group("last4"))

contact_redacted = products["contact"].str.replace(REDACT, redact, regex=True)
print(contact_redacted.head(10))

# A function is applicable to all different variations of phone number and email

0    orders@northwind.example.com
1                  (202) 555-XXXX
2                    202-555-XXXX
3        support@acme.example.org
4                 +1 202 555 XXXX
5          sales@acme.example.org
6                    202.555.XXXX
7                   hello@example
8                                
9                  1-202-555-XXXX
Name: contact, dtype: object


# 5

In [18]:
# Q5.1
NUM = r"(?P<n>\d+(?:\.\d+)?)"
# products["dims"].str.extract(NUM)
products["dims"].str.extractall(NUM)

# str.extract has one dimension per row, str.extractall has 3. The index of the extract all result is 0, 1, 2,
# because there are 3 numbers in each valid row for the dims column

n
   match      
0  0        12
   1         8
   2         3
1  0        12
   1         8
...        ...
58 1        18
   2         6
59 0       1.5
   1       1.5
   2      0.25

[162 rows x 1 columns]

In [19]:
# Q5.2
s = pd.Series(["AB-1042-XY", "ab-6001-yz", None])
ok = s.str.fullmatch(r"[A-Z]{2}-\d{4}-[A-Z]{2}")
print(list(ok), ok.dtype)
# errors 
# print(list(~ok))
# print(list(s[~ok]))

# the second one is false because it is case sensitive. If we add re.IGNORECASE, it would be fine

[True, False, None] object


# 6

In [20]:
# Q6.1
LOG_regex = re.compile(r"""
    ^
    (?P<ip>\S+)                     # client IP
    \s+-\s+-\s+                     # two unused identity fields, always -
    \[(?P<ts>[^\]]+)\]              # timestamp inside [], any char but the closing ]
    \s+"                            # opening quote of the request line
    (?P<method>[A-Z]+)              # method: capitals only
    \s+
    (?P<path>[^"]*)                   # path: no spaces, stops before the protocol
    \s+HTTP/\d\.\d                  # protocol, matched but not captured
    "                               # closing quote of the request line
    \s+(?P<status>\d{3})            # status code: exactly three digits
    \s+(?P<size>\d+|-)              # response size, or - when there is no body
    \s+"(?P<referer>[^"]*)"         # referer, may be empty or "-"
    \s+"(?P<agent>[^"]*)"           # user-agent, may contain spaces and slashes
    $
""", re.VERBOSE)


count = 0
valid = 0
for line in LOG:
    count += 1
    if LOG_regex.fullmatch(line):
        valid += 1
    else:
        print(line)

print(f"valid: {valid}")
print(f"invalid: {count-valid}")

# 192.168.1.24 - - [11/Mar/2026:13:01:59 -0500] "GET /search?q=a"b HTTP/1.1" 200 900 "-" "curl/8.4.0"

# A looser pattern is a worse result because it allows more room for error.

10.0.0.7 - - [11/Mar/2026:09:03:11 -0500] "GET /catalog HTTP/1.1" 200
203.0.113.19 - - [11/Mar/2026:10:22:47 -0500] "GET /cart HTTP/1.1" 200 1120 "-"
<<< log rotated at 11/Mar/2026:11:00:00 >>>
198.51.100.42 - - 11/Mar/2026:12:14:03 -0500 "GET /health HTTP/1.1" 200 12 "-" "curl/8.4.0"
192.168.1.24 - - [11/Mar/2026:13:01:59 -0500] "GET /search?q=a"b HTTP/1.1" 200 900 "-" "curl/8.4.0"
valid: 112
invalid: 5


In [21]:
# Q6.2

# Claude - How should I compare the string "GET /a" and "POST /b"' between ".*", ".*?" and "[^"]*"?

s = '"GET /a" and "POST /b"'
print("subject:", s, "\n")

for name, pat in [("greedy   \".*\"",    r'"(.*)"'),
                  ("lazy     \".*?\"",   r'"(.*?)"'),
                  ("negated  \"[^\"]*\"", r'"([^"]*)"')]:
    print(f"{name:20} matches: {re.findall(pat, s)}")

print("\n--- where lazy and negated diverge ---")
for name, pat in [("lazy    ", r'"(.*?)/b'),
                  ("negated ", r'"([^"]*)/b')]:
    print(f"{name} ->", re.findall(pat, s))
    
# Matches are printed
# I would put negated in a parser

subject: "GET /a" and "POST /b" 

greedy   ".*"        matches: ['GET /a" and "POST /b']
lazy     ".*?"       matches: ['GET /a', 'POST /b']
negated  "[^"]*"     matches: ['GET /a', 'POST /b']

--- where lazy and negated diverge ---
lazy     -> ['GET /a" and "POST ']
negated  -> ['POST ']


In [22]:
# Q6.3

# Claude - Time the pattern (a+)+$ against the strings "a"*n + "!" for n = 14. . . 22.

import time

PAT = re.compile(r"(a+)+$")

for n in range(14, 23):
    s = "a" * n + "!"
    t0 = time.perf_counter()
    PAT.match(s)          # returns None — that is the whole problem
    print(n, time.perf_counter() - t0)

# It is dangerous if the input comes from a user because it will be quite difficult to pin down exact patterns
# that regex would consistently parse correctly. Simple string manipulation is also probably not the best task to
# use regex for. Instead, you should probably use simple string methods for readability

14 0.0031036250293254852
15 0.004418040974996984
16 0.008818334026727825
17 0.0177560830488801
18 0.033788166008889675
19 0.0630543339648284
20 0.11522479203995317
21 0.21817050001118332
22 0.4425107499700971


# Challenge

A colleague could probably act on the fact that there are some duplicates with the same timestamp. This could be problematic in some cases, theoretically.